<h1>Data Cleaning</h1>

In [1]:
import pandas as pd
import numpy as np

profiles_raw = pd.read_csv("../data/profiles.csv")
titles_raw = pd.read_csv("../data/titles.csv")

<h2>Basic cleaning: types & formats</h2>

<h2>Data Cleaning Steps</h2>

<p>
After exploring the data, I moved on to cleaning both the profiles and titles tables.  
Below is a clear summary of the changes made and why each step was necessary.
</p>

<h3>1. Creating Working Copies</h3>
<p>
I created copies of the raw data so that the original files remain untouched.  
This allows safe experimentation and prevents accidental data loss.
</p>

<h3>2. Converting Dates</h3>
<p>
The <strong>created_at</strong> column in the profiles table was converted to a proper datetime format.  
Invalid dates were turned into missing values using <code>errors="coerce"</code>.  
This ensures the column can be used for time analysis later.
</p>

<h3>3. Converting Kids Profile to Boolean</h3>
<p>
The <strong>kids_profile</strong> column was converted to true or false values.  
This makes filtering and logic much simpler when modelling or segmenting users.
</p>

<h3>4. Cleaning Text Columns</h3>
<p>
Several text columns in both tables had unnecessary spaces.  
I removed leading and trailing whitespace to prevent issues like duplicated categories or mismatched profile names.
</p>

<ul>
  <li>Profiles cleaned columns: <em>profile_name, age_band, preferred_language, preferences</em></li>
  <li>Titles cleaned columns: <em>title_name, category, sub_category, age_rating, type, origin_region, language</em></li>
</ul>

<h3>5. Converting Numeric Columns</h3>
<p>
Some numeric columns were stored as strings.  
I converted them to numeric types so they can be used for calculations and modelling.  
Invalid values become NaN using <code>errors="coerce"</code>.
</p>

<ul>
  <li><strong>duration</strong></li>
  <li><strong>year</strong></li>
  <li><strong>episode_count</strong></li>
  <li><strong>imdb_rating</strong></li>
  <li><strong>imdb_votes</strong></li>
</ul>

<h3>6. Converting Kids Content Flag</h3>
<p>
The <strong>is_kids_content</strong> column in the titles table was also converted to true or false values.  
This ensures consistency with the kids profile flag and helps with filtering recommendations.
</p>

<p>
Overall, these cleaning steps make the dataset consistent, structured, and ready for the next stage of analysis and modelling.
</p>

In [2]:
profiles = profiles_raw.copy()
titles = titles_raw.copy()

# created_at to datetime
profiles["created_at"] = pd.to_datetime(profiles["created_at"], errors = "coerce")

# kids_profile to bool
profiles["kids_profile"] = profiles["kids_profile"].astype(bool)

# strip whitespace from strings
str_cols_profiles = ["profile_name", "age_band", "preferred_language", "preferences"]
for col in str_cols_profiles:
    profiles[col] = profiles[col].astype(str).str.strip()

str_cols_titles = ["title_name", "category", "sub_category", "age_rating", "type", "origin_region", "language"]

for col in str_cols_titles:
    titles[col] = titles[col].astype(str).str.strip()

# numeric conversions
titles["duration"] = pd.to_numeric(titles["duration"], errors="coerce")
titles["year"] = pd.to_numeric(titles["year"], errors="coerce")
titles["episode_count"] = pd.to_numeric(titles["episode_count"], errors="coerce")
titles["imdb_rating"] = pd.to_numeric(titles["imdb_rating"], errors="coerce")
titles["imdb_votes"] = pd.to_numeric(titles["imdb_votes"], errors="coerce")

# is_kids_content to bool
titles["is_kids_content"] = titles["is_kids_content"].astype(bool)


<h2>Deduplication and Missing Value Handling</h2>

<p>
After the initial cleaning, I focused on removing duplicates and handling missing values.  
These steps make the data more reliable for analysis and modelling.
</p>

<h3>1. Removing Duplicate Profiles</h3>
<p>
The <strong>profiles</strong> table was checked for duplicate entries using the <strong>profile_id</strong>.  
Any duplicate profiles were removed so that each profile is unique.
</p>

<h3>2. Removing Duplicate Titles</h3>
<p>
The <strong>titles</strong> table was checked for duplicates using the <strong>show_id</strong>.  
Duplicate titles were removed to ensure each show only appears once in the dataset.
</p>

<h3>3. Dropping Titles with Missing Names</h3>
<p>
Some titles had missing or empty <strong>title_name</strong> values.  
These rows were removed because a missing name makes the title unusable for recommendations.
</p>

<h3>4. Handling Missing Preferred Language</h3>
<p>
If any profile had a missing <strong>preferred_language</strong>, it was set to <strong>English</strong> by default.  
This ensures that all profiles have a valid language for filtering and recommendation logic.
</p>

<p>
These steps improve data consistency and ensure that the dataset is ready for the recommendation modelling phase.
</p>

<h2> Deduplicate & handle obvious nulls </h2>

In [3]:
# deduplicate profiles on profile_id
profiles = profiles.drop_duplicates(subset=["profile_id"])

# deduplicate titles on show_id
titles = titles.drop_duplicates(subset=["show_id"])

# drop titles where title_name is missing or empty
titles = titles[~titles["title_name"].isna() & (titles["title_name"].str.len() > 0)]

# if any preffered_language missing, default to 'English'
profiles["preferred_language"] = profiles["preferred_language"].replace("nan", np.nan)
profiles["preferred_language"] = profiles["preferred_language"].fillna("English")



<h1> Extract genres from preferences </h1>
<h5> Here I am normalizing the preferences into a list of genres to map to category/sub_category </h5>

In [6]:
def parse_preferences(pref_str):
    if pd.isna(pref_str) or pref_str.strip() == "":
        return []
    return [p.strip() for p in str(pref_str).split(",")]

profiles["preference_list"] = profiles["preferences"].apply(parse_preferences)
profiles[["profile_id", "preferences", "preference_list"]].head()

,profile_id,preferences,preference_list
0,1,"Horror, Thriller","[Horror, Thriller]"
1,2,Horror,[Horror]
2,3,"Horror, Thriller","[Horror, Thriller]"
3,4,"Animation, Family","[Animation, Family]"
4,5,"Comedy, Action, Game-Show","[Comedy, Action, Game-Show]"


<h2>Parsing Profile Preferences</h2>

<p>
Profiles contain a <strong>preferences</strong> column that stores user genre preferences as comma-separated strings.  
To make this data easier to use for recommendations, I converted each string into a proper Python list.
</p>

<h3>1. Parsing Function</h3>
<p>
I defined a function called <code>parse_preferences</code> that:
</p>
<ul>
  <li>Checks if the value is missing or empty. If so, it returns an empty list.</li>
  <li>Otherwise, splits the string on commas and removes extra whitespace from each genre.</li>
</ul>

<h3>2. Applying the Function</h3>
<p>
The function is applied to the <strong>preferences</strong> column to create a new column called <strong>preference_list</strong>.  
This column now contains a list of preferences for each profile instead of a single string.
</p>

<h3>3. Example Output</h3>
<pre>
<code>
profile_id | preferences            | preference_list
-----------|-----------------------|--------------------------
1          | "Drama, Comedy"       | ["Drama", "Comedy"]
2          | "Action"              | ["Action"]
3          | NaN                   | []
</code>
</pre>

<p>
This transformation makes it much easier to check for genre matches when generating recommendations.
</p>

<h3> Final sanity checks </h3>

<h2>Checking Data Shape and Null Values</h2>

<p>
After cleaning and transforming the data, it’s important to confirm the dataset sizes and check for any remaining null values.
</p>

<h3>1. Checking Null Values</h3>
<p>
We can check for missing values in both tables using <code>isna().sum()</code>.  
This step ensures we know if there are still missing values that need handling.
</p>

<h3>2. Checking Dataset Shape</h3>
<p>
The shape of each table shows the number of rows and columns remaining after cleaning:
</p>

<pre>
<code>
Profiles shape: (number_of_profiles, number_of_columns)
Titles shape:   (number_of_titles, number_of_columns)
</code>
</pre>

<p>
This confirms that duplicates and invalid entries have been removed, and the datasets are ready for modelling and recommendations.
</p>

In [5]:
print("Profiles nulls:\n", profiles.isna().sum())
print("\nTitles nulls:\n", titles.isna().sum())

print("\nProfiles shape:", profiles.shape)
print("Titles shape:", titles.shape)

Profiles nulls:
 profile_id            0
account_id            0
profile_name          0
kids_profile          0
age_band              0
preferred_language    0
created_at            0
preferences           0
dtype: int64

Titles nulls:
 show_id              0
title_name           0
category             0
sub_category         0
duration             0
age_rating           0
type                 0
year                 0
origin_region        0
language             0
episode_count        0
is_kids_content      0
imdb_rating        368
imdb_votes         368
dtype: int64

Profiles shape: (359, 8)
Titles shape: (1000, 14)


In [ ]:
# save clean versions
profiles.to_csv("../data/profiles_clean.csv", index=False)
titles.to_csv("../data/titles_clean.csv", index=False)